In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("ps4_2_data.csv")
df.head()

,Outlook,Temperature,Humidity,Wind,PlayTennis
0,Sunny,30,High,Weak,No
1,Sunny,32,High,Strong,No
2,Overcast,28,High,Weak,Yes
3,Rain,25,High,Weak,Yes
4,Rain,22,Normal,Weak,Yes


In [5]:
target = df.columns[-1]
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

classes = y.unique()

In [6]:
numerical_cols = []
categorical_cols = []
for col in X.columns:
    if np.issubdtype(X[col].dtype,np.number):
        numerical_cols.append(col)
    else:
        categorical_cols.append(col)

print(numerical_cols)
print(categorical_cols)

['Temperature']
['Outlook', 'Humidity', 'Wind']


In [ ]:
prior = {}
for c in classes:
    prior[c] = len(y[ y == c])/len(y)

In [ ]:
cat_probs = {}
for col in categorical_cols:
    cat_probs[col] = {}
    categories = X[col].unique()
    k = len(categories)
    for c in classes:
        subset = df[df[target] == c]
        counts = subset[col].value_counts()
        prob = {}
        for category in categories:
            cat_count = counts.get(category,0)
            val = (cat_count + 1)/(len(subset) + k) #laplacian smoothing
            prob[category] = val

        cat_probs[col][c] = prob

cat_probs

{'Outlook': {'No': {'Sunny': 0.5, 'Overcast': 0.125, 'Rain': 0.375},
  'Yes': {'Sunny': 0.25,
   'Overcast': 0.4166666666666667,
   'Rain': 0.3333333333333333}},
 'Humidity': {'No': {'High': 0.7142857142857143, 'Normal': 0.2857142857142857},
  'Yes': {'High': 0.36363636363636365, 'Normal': 0.6363636363636364}},
 'Wind': {'No': {'Weak': 0.42857142857142855, 'Strong': 0.5714285714285714},
  'Yes': {'Weak': 0.6363636363636364, 'Strong': 0.36363636363636365}}}

In [11]:
num_stats = {}
for col in numerical_cols:
    num_stats[col] = {}
    for c in classes:
        subset = df[df[target] == c]
        mean = subset[col].mean()
        std = subset[col].std()
        if std == 0:
            std = 1e-6
        num_stats[col][c] = (mean,std)

num_stats

{'Temperature': {'No': (25.6, 5.856620185738529),
  'Yes': (25.444444444444443, 3.3582402799349804)}}

In [12]:
def gaussian_prob(x,mean,std):
    e = np.exp(-(x-mean)**2/(2*std**2))
    return (1/(std*np.sqrt(2*np.pi)))*e

In [16]:
def predict(sample):
    posteriors = {}
    for c in classes:
        prob = np.log(prior[c])
        for col in categorical_cols:
            value = sample[col]
            prob += np.log(cat_probs[col][c].get(value,1e-6))
        for col in numerical_cols:
            mean,std = num_stats[col][c]
            prob += np.log(gaussian_prob(sample[col],mean,std))
        posteriors[c] = prob
    
    return max(posteriors,key = posteriors.get)


In [17]:
sample = {
    "Outlook": "Sunny",
    "Temperature": 26,
    "Humidity": "High",
    "Wind": "Weak"
}

predicted = predict(sample)
print(sample)
print(predicted)

{'Outlook': 'Sunny', 'Temperature': 26, 'Humidity': 'High', 'Wind': 'Weak'}
Yes
